Mini-Project : Data Analysis for Marketing Strategy

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# DATA LOADING AND PREPROCESSING
df = pd.read_csv("US_Superstore_data.xls - Orders.csv")

df['Order_Date_Clean'] = pd.to_datetime(df['Order Date'], unit='D', origin='1899-12-30')
df['Ship_Date_Clean'] = pd.to_datetime(df['Ship Date'], unit='D', origin='1899-12-30')
df['Year'] = df['Order_Date_Clean'].dt.year

print("Data loaded successfully. Cleaned dataset sample:")
print(df[['Order ID', 'Order_Date_Clean', 'Sales', 'Profit']].head())


# Q1 - Which states have the most sales?
state_sales = df.groupby('State')['Sales'].sum().reset_index().sort_values(by='Sales', ascending=False)

plt.figure(figsize=(12, 5))
sns.barplot(data=state_sales.head(10), x='Sales', y='State', palette='viridis', hue='State', legend=False)
plt.title('Top 10 US States by Total Sales Volume')
plt.xlabel('Total Sales ($)')
plt.ylabel('State')
plt.tight_layout()
plt.show()

# Insight: California and New York generate the highest total revenue out of all US states.


# Q2 - Difference between New York and California (Sales & Profit)
ny_ca = df[df['State'].isin(['New York', 'California'])].groupby('State')[['Sales', 'Profit']].sum().reset_index()
ny_ca_melted = ny_ca.melt(id_vars='State', value_vars=['Sales', 'Profit'], var_name='Metric', value_name='Amount')

plt.figure(figsize=(8, 5))
sns.barplot(data=ny_ca_melted, x='State', y='Amount', hue='Metric', palette='Set2')
plt.title('Sales and Profit Comparison: California vs. New York')
plt.ylabel('Amount ($)')
plt.xlabel('State')
plt.tight_layout()
plt.show()

# Insight: California drives around 47% more revenue than New York. However, both states
# finish with nearly identical net profits (~$76k vs ~$74k), making New York more cost-efficient.


# Q3 - Outstanding customer in New York
ny_customers = df[df['State'] == 'New York'].groupby(['Customer ID', 'Customer Name'])[['Sales', 'Profit']].sum().reset_index()
top_ny_sales = ny_customers.sort_values(by='Sales', ascending=False).head(5)

plt.figure(figsize=(8, 4))
sns.barplot(data=top_ny_sales, x='Sales', y='Customer Name', palette='Blues_r', hue='Customer Name', legend=False)
plt.title('Top 5 Customers by Sales in New York')
plt.xlabel('Sales ($)')
plt.show()

# Insight: Tom Ashbrook is the standout customer in New York, ranking #1 in sales ($13,723.50)
# as well as net profit contribution ($4,599.21).


# Q4 - Differences among states in profitability
state_profit = df.groupby('State')['Profit'].sum().reset_index().sort_values(by='Profit', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(data=state_profit, x='Profit', y='State', palette='coolwarm', hue='State', legend=False)
plt.title('Total Net Profit / Loss Distribution by State')
plt.xlabel('Net Profit ($)')
plt.ylabel('State')
plt.tight_layout()
plt.show()

# Insight: High sales do not guarantee profits. States like Texas, Ohio, and Pennsylvania
# face extreme financial losses due to deep promotional price discounting.


# Q5 - Pareto Principle (80/20 rule) applied to customers and Profit
cust_profit = df.groupby('Customer ID')['Profit'].sum().reset_index().sort_values(by='Profit', ascending=False).reset_index(drop=True)
cust_profit['Cum_Profit'] = cust_profit['Profit'].cumsum()
cust_profit['Cum_Profit_Pct'] = cust_profit['Cum_Profit'] / cust_profit['Profit'].sum()

idx_20 = int(round(0.20 * len(cust_profit))) - 1
pct_profit_20 = cust_profit.loc[idx_20, 'Cum_Profit_Pct'] * 100

plt.figure(figsize=(9, 5))
plt.plot(cust_profit.index / len(cust_profit) * 100, cust_profit['Cum_Profit_Pct'] * 100, color='g', linewidth=2)
plt.axvline(20, color='r', linestyle='--', label='20% of Customers')
plt.axhline(pct_profit_20, color='blue', linestyle='--', label=f'{pct_profit_20:.2f}% of Profits')
plt.title('Cumulative Profit Curve by Customer (Pareto Analysis)')
plt.xlabel('% of Customer Base')
plt.ylabel('% of Total Accumulated Profits')
plt.legend()
plt.show()

# Conclusion: YES, the Pareto Principle applies perfectly. The top 20% of customers generate 81.66% of profits.


# Q6 - Top 20 cities by Sales vs Profit & Profitability Differences
city_stats = df.groupby('City')[['Sales', 'Profit']].sum().reset_index()
top_20_city_sales = city_stats.sort_values(by='Sales', ascending=False).head(20)
top_20_city_profit = city_stats.sort_values(by='Profit', ascending=False).head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
sns.barplot(data=top_20_city_sales, x='Sales', y='City', ax=axes[0], palette='Purples_r', hue='City', legend=False)
axes[0].set_title('Top 20 Cities by Total Sales Volume')

sns.barplot(data=top_20_city_profit, x='Profit', y='City', ax=axes[1], palette='Greens_r', hue='City', legend=False)
axes[1].set_title('Top 20 Cities by Net Profit')
plt.tight_layout()
plt.show()

# Insight: High revenue cities like Philadelphia and Houston generate heavy structural losses,
# whereas New York City, Los Angeles, and Seattle are highly secure profit centers.


# Q7 - Top 20 customers by Sales
top_cust_sales = df.groupby('Customer Name')['Sales'].sum().reset_index().sort_values(by='Sales', ascending=False).head(20)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_cust_sales, x='Sales', y='Customer Name', palette='plasma', hue='Customer Name', legend=False)
plt.title('Top 20 Store Customers by Global Revenue Generation')
plt.xlabel('Total Revenue Contributed ($)')
plt.ylabel('Customer Name')
plt.tight_layout()
plt.show()

# Insight: Sean Miller, Tamara Chand, and Raymond Buch are the top global revenue contributors.

# Q8 - Cumulative Curve in Sales by Customers & Pareto for Sales
cust_sales = df.groupby('Customer ID')['Sales'].sum().reset_index().sort_values(by='Sales', ascending=False).reset_index(drop=True)
cust_sales['Cum_Sales'] = cust_sales['Sales'].cumsum()
cust_sales['Cum_Sales_Pct'] = cust_sales['Cum_Sales'] / cust_sales['Sales'].sum()

pct_sales_20 = cust_sales.loc[idx_20, 'Cum_Sales_Pct'] * 100

plt.figure(figsize=(9, 5))
plt.plot(cust_sales.index / len(cust_sales) * 100, cust_sales['Cum_Sales_Pct'] * 100, color='orange', linewidth=2)
plt.axvline(20, color='r', linestyle='--', label='20% of Customers')
plt.axhline(pct_sales_20, color='blue', linestyle='--', label=f'{pct_sales_20:.2f}% of Sales')
plt.title('Cumulative Sales Curve by Customer')
plt.xlabel('% of Customer Base')
plt.ylabel('% of Total Accumulated Sales')
plt.legend()
plt.show()

# Conclusion: NO, the Pareto Principle does not apply to sales. The top 20% of customers 
# account for only 48.15% of sales volume, showing purchasing behavior is more evenly spread.


# Q9 - Strategic Marketing Decisions
print("""
=== STRATEGIC TARGETING DIRECTIVES ===
1. PRIMARY ACQUISITION EXPANSION (High Volume, High Margin):
   - Targets: New York City (NY), Los Angeles (CA), Seattle (WA), and San Francisco (CA).
   - Action: Focus active growth budgets here. New York City delivers exceptional efficiency.

2. CORRECTION AND COST MITIGATION (High Volume, Heavy Losses):
   - Targets: Texas (Houston/San Antonio) and Pennsylvania (Philadelphia).
   - Action: Cease aggressive general marketing and stop wide-scale promotional discounts. 
     Pivot sales models strictly toward high-margin sub-categories to recover loss.

3. LOYALTY RETENTION PIPELINE (CRM Target):
   - Action: Deploy dedicated VIP account managers for the top 20% profitable customers. 
     Since they protect over 81% of total earnings, preserving these relations is mandatory.
""")